In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import math

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 3, Finished, Available, Finished, False)

In [2]:
gold_fact_sales = spark.table("gold_fact_sales")

print("Gold fact loaded.")
gold_fact_sales.printSchema()

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 4, Finished, Available, Finished, False)

Gold fact loaded.
root
 |-- date_key: integer (nullable = true)
 |-- product_key: integer (nullable = true)
 |-- store_key: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- sales: integer (nullable = true)
 |-- sell_price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- weekday: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- event_name_1: string (nullable = true)
 |-- event_type_1: string (nullable = true)
 |-- event_name_2: string (nullable = true)
 |-- event_type_2: string (nullable = true)



In [3]:
date_range = gold_fact_sales.select(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

print("Minimum date:", min_date)
print("Maximum date:", max_date)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 5, Finished, Available, Finished, False)

Minimum date: 2011-01-29
Maximum date: 2016-04-24


In [4]:
BACKTEST_DAYS = 28

backtest_cutoff = (
    gold_fact_sales
    .select(F.date_sub(F.max("date"), BACKTEST_DAYS).alias("cutoff"))
    .collect()[0]["cutoff"]
)

print("Backtest cutoff:", backtest_cutoff)
print("Validation days:", BACKTEST_DAYS)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 6, Finished, Available, Finished, False)

Backtest cutoff: 2016-03-27
Validation days: 28


In [5]:
daily_demand = (
    gold_fact_sales
    .select(
        "date",
        "item_id",
        "store_id",
        "sales"
    )
    .groupBy(
        "date",
        "item_id",
        "store_id"
    )
    .agg(
        F.sum("sales").alias("sales")
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 7, Finished, Available, Finished, False)

In [6]:
print("Daily demand rows:", daily_demand.count())
print(
    "Unique item-store series:",
    daily_demand.select("item_id", "store_id").distinct().count()
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 8, Finished, Available, Finished, False)

Daily demand rows: 58327370
Unique item-store series: 30490


In [7]:
lag_window = (
    Window
    .partitionBy("item_id", "store_id")
    .orderBy("date")
)

forecast_features = (
    daily_demand
    .withColumn("lag_7", F.lag("sales", 7).over(lag_window))
    .withColumn("lag_14", F.lag("sales", 14).over(lag_window))
    .withColumn("lag_21", F.lag("sales", 21).over(lag_window))
    .withColumn("lag_28", F.lag("sales", 28).over(lag_window))
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 9, Finished, Available, Finished, False)

In [8]:
forecast_features = (
    forecast_features
    .withColumn(
        "seasonal_forecast",
        F.round(
            (
                F.col("lag_7") +
                F.col("lag_14") +
                F.col("lag_21") +
                F.col("lag_28")
            ) / 4,
            2
        )
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 10, Finished, Available, Finished, False)

In [9]:
display(
    forecast_features
    .filter(F.col("seasonal_forecast").isNotNull())
    .limit(20)
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1fdcea8a-8494-4fe7-8331-ee9e4412b66d)

In [10]:
backtest_predictions = (
    forecast_features
    .filter(
        F.col("date") > F.lit(backtest_cutoff)
    )
    .filter(
        F.col("seasonal_forecast").isNotNull()
    )
    .select(
        "date",
        "item_id",
        "store_id",
        F.col("sales").alias("actual_sales"),
        F.col("seasonal_forecast").alias("predicted_sales")
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 12, Finished, Available, Finished, False)

In [11]:
mae = (
    backtest_predictions
    .select(
        F.avg(
            F.abs(
                F.col("actual_sales") -
                F.col("predicted_sales")
            )
        ).alias("MAE")
    )
    .collect()[0]["MAE"]
)

print(f"Backtest MAE: {mae:.4f}")

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 13, Finished, Available, Finished, False)

Backtest MAE: 1.0463


In [12]:
rmse = (
    backtest_predictions
    .select(
        F.sqrt(
            F.avg(
                F.pow(
                    F.col("actual_sales") -
                    F.col("predicted_sales"),
                    2
                )
            )
        ).alias("RMSE")
    )
    .collect()[0]["RMSE"]
)

print(f"Backtest RMSE: {rmse:.4f}")

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 14, Finished, Available, Finished, False)

Backtest RMSE: 2.1880


In [13]:
wape_values = (
    backtest_predictions
    .select(
        F.sum(
            F.abs(
                F.col("actual_sales") -
                F.col("predicted_sales")
            )
        ).alias("absolute_error"),

        F.sum("actual_sales").alias("actual_demand")
    )
    .collect()[0]
)

wape = (
    wape_values["absolute_error"] /
    wape_values["actual_demand"]
) * 100

print(f"Backtest WAPE: {wape:.2f}%")

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 15, Finished, Available, Finished, False)

Backtest WAPE: 75.46%


In [14]:
display(
    backtest_predictions
    .orderBy("date")
    .limit(100)
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e5e952a0-a9fc-49ea-9a7c-3135a213202a)

In [15]:
series = (
    daily_demand
    .select(
        "item_id",
        "store_id"
    )
    .distinct()
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 17, Finished, Available, Finished, False)

In [16]:
future_dates = (
    spark.range(1, 29)
    .withColumn(
        "date",
        F.date_add(
            F.lit(max_date),
            F.col("id").cast("int")
        )
    )
    .drop("id")
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 18, Finished, Available, Finished, False)

In [17]:
display(future_dates)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c3e7ac97-89e8-4282-bdf0-b79620c68332)

In [18]:
future_grid = (
    series
    .crossJoin(future_dates)
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 20, Finished, Available, Finished, False)

In [19]:
sales_7 = daily_demand.select(
    "item_id",
    "store_id",
    F.date_add(F.col("date"), 7).alias("future_date"),
    F.col("sales").alias("sales_7")
)

sales_14 = daily_demand.select(
    "item_id",
    "store_id",
    F.date_add(F.col("date"), 14).alias("future_date"),
    F.col("sales").alias("sales_14")
)

sales_21 = daily_demand.select(
    "item_id",
    "store_id",
    F.date_add(F.col("date"), 21).alias("future_date"),
    F.col("sales").alias("sales_21")
)

sales_28 = daily_demand.select(
    "item_id",
    "store_id",
    F.date_add(F.col("date"), 28).alias("future_date"),
    F.col("sales").alias("sales_28")
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 21, Finished, Available, Finished, False)

In [20]:
future_forecast = (
    future_grid
    .withColumnRenamed("date", "future_date")
    .join(
        sales_7,
        on=["item_id", "store_id", "future_date"],
        how="left"
    )
    .join(
        sales_14,
        on=["item_id", "store_id", "future_date"],
        how="left"
    )
    .join(
        sales_21,
        on=["item_id", "store_id", "future_date"],
        how="left"
    )
    .join(
        sales_28,
        on=["item_id", "store_id", "future_date"],
        how="left"
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 22, Finished, Available, Finished, False)

In [21]:
future_forecast.printSchema()
display(future_forecast.limit(10))

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 23, Finished, Available, Finished, False)

root
 |-- item_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- future_date: date (nullable = false)
 |-- sales_7: long (nullable = true)
 |-- sales_14: long (nullable = true)
 |-- sales_21: long (nullable = true)
 |-- sales_28: long (nullable = true)



SynapseWidget(Synapse.DataFrame, 1d156a72-6cae-45e9-956c-d6cf8257931a)

In [22]:
future_forecast = (
    future_forecast
    .withColumn(
        "demand_forecast",
        F.round(
            (
                F.coalesce(F.col("sales_7"), F.lit(0)) +
                F.coalesce(F.col("sales_14"), F.lit(0)) +
                F.coalesce(F.col("sales_21"), F.lit(0)) +
                F.coalesce(F.col("sales_28"), F.lit(0))
            ) / 4,
            2
        )
    )
)
future_forecast = (
    future_forecast
    .select(
        "future_date",
        "item_id",
        "store_id",
        "demand_forecast",
        "sales_7",
        "sales_14",
        "sales_21",
        "sales_28"
    )
)


StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 24, Finished, Available, Finished, False)

In [23]:
display(
    future_forecast
    .orderBy("future_date", "store_id", "item_id")
    .limit(30)
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b88f4924-f485-471f-95e3-fe30c9028b9e)

In [24]:
future_forecast = (
    future_forecast
    .withColumn(
        "demand_stddev",
        F.sqrt(
            (
                F.pow(
                    F.coalesce(F.col("sales_7"), F.lit(0)) -
                    F.col("demand_forecast"),
                    2
                ) +
                F.pow(
                    F.coalesce(F.col("sales_14"), F.lit(0)) -
                    F.col("demand_forecast"),
                    2
                ) +
                F.pow(
                    F.coalesce(F.col("sales_21"), F.lit(0)) -
                    F.col("demand_forecast"),
                    2
                ) +
                F.pow(
                    F.coalesce(F.col("sales_28"), F.lit(0)) -
                    F.col("demand_forecast"),
                    2
                )
            ) / 4
        )
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 26, Finished, Available, Finished, False)

In [25]:
future_forecast = (
    future_forecast
    .withColumn(
        "safety_stock_proxy",
        F.round(
            1.65 * F.coalesce(
                F.col("demand_stddev"),
                F.lit(0)
            ),
            2
        )
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 27, Finished, Available, Finished, False)

In [26]:
future_forecast = (
    future_forecast
    .withColumn(
        "replenishment_target",
        F.round(
            F.col("demand_forecast") +
            F.col("safety_stock_proxy"),
            2
        )
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 28, Finished, Available, Finished, False)

In [27]:
future_forecast = (
    future_forecast
    .withColumn(
        "demand_pressure",
        F.when(
            F.col("demand_forecast") > 0,
            F.col("safety_stock_proxy") /
            F.col("demand_forecast")
        ).otherwise(0)
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 29, Finished, Available, Finished, False)

In [28]:
percentiles = future_forecast.select(
    F.expr("percentile_approx(demand_forecast, 0.95)").alias("p95_demand"),
    F.expr("percentile_approx(demand_stddev, 0.95)").alias("p95_stddev")
).collect()[0]

p95_demand = percentiles["p95_demand"]
p95_stddev = percentiles["p95_stddev"]

print("95th percentile demand:", p95_demand)
print("95th percentile volatility:", p95_stddev)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 30, Finished, Available, Finished, False)

95th percentile demand: 3.5
95th percentile volatility: 3.031088913245535


In [29]:
future_forecast = (
    future_forecast
    .withColumn(
        "demand_risk_component",
        F.least(
            F.col("demand_forecast") / F.lit(p95_demand),
            F.lit(1.0)
        )
    )
    .withColumn(
        "volatility_risk_component",
        F.least(
            F.col("demand_stddev") / F.lit(p95_stddev),
            F.lit(1.0)
        )
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 31, Finished, Available, Finished, False)

In [30]:
future_forecast = (
    future_forecast
    .withColumn(
        "inventory_risk_score",
        F.round(
            (
                F.col("demand_risk_component") * 40 +
                F.col("volatility_risk_component") * 60
            ),
            2
        )
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 32, Finished, Available, Finished, False)

In [31]:
future_forecast = (
    future_forecast
    .withColumn(
        "inventory_risk_category",
        F.when(
            F.col("inventory_risk_score") >= 70,
            "HIGH"
        )
        .when(
            F.col("inventory_risk_score") >= 40,
            "MEDIUM"
        )
        .otherwise("LOW")
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 33, Finished, Available, Finished, False)

In [32]:
from pyspark.sql import functions as F

gold_dim_product = spark.table("gold_dim_product")
gold_dim_store = spark.table("gold_dim_store")

print("Gold dimensions loaded successfully.")

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 34, Finished, Available, Finished, False)

Gold dimensions loaded successfully.


In [33]:
future_forecast = (
    future_forecast
    .join(
        gold_dim_product
        .select("item_id", "dept_id", "cat_id")
        .dropDuplicates(["item_id"]),
        on="item_id",
        how="left"
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 35, Finished, Available, Finished, False)

In [34]:
future_forecast = (
    future_forecast
    .join(
        gold_dim_store
        .select("store_id", "state_id")
        .dropDuplicates(["store_id"]),
        on="store_id",
        how="left"
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 36, Finished, Available, Finished, False)

In [35]:
future_forecast.printSchema()
display(future_forecast.limit(10))

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 37, Finished, Available, Finished, False)

root
 |-- store_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- future_date: date (nullable = false)
 |-- demand_forecast: double (nullable = true)
 |-- sales_7: long (nullable = true)
 |-- sales_14: long (nullable = true)
 |-- sales_21: long (nullable = true)
 |-- sales_28: long (nullable = true)
 |-- demand_stddev: double (nullable = true)
 |-- safety_stock_proxy: double (nullable = true)
 |-- replenishment_target: double (nullable = true)
 |-- demand_pressure: double (nullable = true)
 |-- demand_risk_component: double (nullable = false)
 |-- volatility_risk_component: double (nullable = false)
 |-- inventory_risk_score: double (nullable = true)
 |-- inventory_risk_category: string (nullable = false)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- state_id: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 29084eff-1cd5-47a2-9fcc-acab7c93bc53)

In [36]:
gold_demand_forecast = (
    future_forecast
    .withColumnRenamed(
        "future_date",
        "forecast_date"
    )
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 38, Finished, Available, Finished, False)

In [37]:
gold_demand_forecast.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_demand_forecast")

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 39, Finished, Available, Finished, False)

In [38]:
print(
    "Forecast rows:",
    gold_demand_forecast.count()
)

print(
    "Forecast dates:",
    gold_demand_forecast
    .select("forecast_date")
    .distinct()
    .count()
)

print(
    "Item-store series:",
    gold_demand_forecast
    .select("item_id", "store_id")
    .distinct()
    .count()
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 40, Finished, Available, Finished, False)

Forecast rows: 853720
Forecast dates: 28
Item-store series: 30490


In [39]:
gold_demand_forecast.select(
    F.min("forecast_date").alias("forecast_start"),
    F.max("forecast_date").alias("forecast_end")
).show()

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 41, Finished, Available, Finished, False)

+--------------+------------+
|forecast_start|forecast_end|
+--------------+------------+
|    2016-04-25|  2016-05-22|
+--------------+------------+



In [40]:
display(
    gold_demand_forecast
    .filter(
        F.col("inventory_risk_category") == "HIGH"
    )
    .orderBy(
        F.desc("inventory_risk_score")
    )
    .select(
        "forecast_date",
        "item_id",
        "store_id",
        "dept_id",
        "cat_id",
        "demand_forecast",
        "safety_stock_proxy",
        "replenishment_target",
        "inventory_risk_score",
        "inventory_risk_category"
    )
    .limit(50)
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d7bd6464-fd7a-4940-9e18-a1eb34eeb5a6)

In [41]:
risk_summary = (
    gold_demand_forecast
    .groupBy("inventory_risk_category")
    .agg(
        F.count("*").alias("forecast_rows"),
        F.countDistinct("item_id").alias("unique_items"),
        F.sum("demand_forecast").alias("forecasted_units")
    )
    .orderBy("inventory_risk_category")
)

display(risk_summary)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 84a48d59-4f63-4781-be82-a2cd120ed322)

In [42]:
import mlflow

mlflow.set_experiment("Retail_Demand_Forecasting")

with mlflow.start_run(run_name="Seasonal_4Week_Forecast"):

    mlflow.log_param(
        "forecast_horizon_days",
        28
    )

    mlflow.log_param(
        "seasonal_lags",
        "7,14,21,28"
    )

    mlflow.log_metric(
        "MAE",
        float(mae)
    )

    mlflow.log_metric(
        "RMSE",
        float(rmse)
    )

    mlflow.log_metric(
        "WAPE_percent",
        float(wape)
    )

    mlflow.log_param(
        "forecast_method",
        "4-week seasonal mean"
    )

    print("MLflow run logged successfully.")

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 44, Finished, Available, Finished, False)

MLflow run logged successfully.


In [43]:
print("NULL forecasts:")

print(
    gold_demand_forecast
    .filter(F.col("demand_forecast").isNull())
    .count()
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 45, Finished, Available, Finished, False)

NULL forecasts:
0


In [44]:
print("Negative forecasts:")

print(
    gold_demand_forecast
    .filter(F.col("demand_forecast") < 0)
    .count()
)

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 46, Finished, Available, Finished, False)

Negative forecasts:
0


In [45]:
print("Risk category distribution:")

gold_demand_forecast.groupBy(
    "inventory_risk_category"
).count().show()

StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 47, Finished, Available, Finished, False)

Risk category distribution:
+-----------------------+------+
|inventory_risk_category| count|
+-----------------------+------+
|                   HIGH| 68447|
|                    LOW|699213|
|                 MEDIUM| 86060|
+-----------------------+------+



In [46]:

from pyspark.sql import functions as F
from datetime import timedelta
import math

print("=" * 65)
print("DAY 5 — DEMAND FORECASTING & INVENTORY INTELLIGENCE AUDIT")
print("=" * 65)

# 1. Check required tables
required_tables = [
    "gold_dim_date",
    "gold_dim_product",
    "gold_dim_store",
    "gold_fact_sales",
    "gold_data_quality_audit",
    "gold_demand_forecast"
]

available_tables = {
    row["tableName"]
    for row in spark.sql("SHOW TABLES").collect()
}

missing_tables = [t for t in required_tables if t not in available_tables]

for table in required_tables:
    print(f"[{'PASS' if table in available_tables else 'FAIL'}] Table: {table}")

if "gold_demand_forecast" not in available_tables:
    raise RuntimeError(
        "FAIL: gold_demand_forecast does not exist. "
        "Complete Part 34 before running this audit."
    )

# 2. Load the persisted forecast
forecast = spark.table("gold_demand_forecast")

required_columns = [
    "forecast_date", "item_id", "store_id",
    "demand_forecast", "demand_stddev",
    "safety_stock_proxy", "replenishment_target",
    "inventory_risk_score", "inventory_risk_category"
]

missing_columns = [
    c for c in required_columns if c not in forecast.columns
]

if missing_columns:
    raise RuntimeError(f"Missing forecast columns: {missing_columns}")

# 3. Check row count, date horizon and item-store coverage
summary = forecast.agg(
    F.count("*").alias("rows"),
    F.countDistinct("forecast_date").alias("dates"),
    F.countDistinct(F.struct("item_id", "store_id")).alias("series"),
    F.min("forecast_date").alias("start_date"),
    F.max("forecast_date").alias("end_date")
).first()

row_count = summary["rows"]
date_count = summary["dates"]
series_count = summary["series"]
start_date = summary["start_date"]
end_date = summary["end_date"]

print("\n--- FORECAST COVERAGE ---")
print(f"Rows:             {row_count:,}")
print(f"Forecast dates:   {date_count}")
print(f"Item-store series:{series_count:>10,}")
print(f"Start date:       {start_date}")
print(f"End date:         {end_date}")

horizon_pass = (
    date_count == 28
    and start_date is not None
    and end_date is not None
    and (end_date - start_date).days == 27
)

print(f"[{'PASS' if horizon_pass else 'FAIL'}] 28 consecutive forecast dates")

# Compare forecast start against historical maximum date
historical_max = (
    spark.table("gold_fact_sales")
    .agg(F.max("date").alias("max_date"))
    .first()["max_date"]
)

date_alignment_pass = (
    historical_max is not None
    and start_date == historical_max + timedelta(days=1)
)

print(f"Historical last date: {historical_max}")
print(
    f"[{'PASS' if date_alignment_pass else 'FAIL'}] "
    "Forecast starts the day after historical data"
)

# 4. Validate forecast values and risk fields
numeric_checks = forecast.agg(
    F.sum(
        F.when(F.col("demand_forecast").isNull(), 1).otherwise(0)
    ).alias("null_forecasts"),
    F.sum(
        F.when(
            F.col("demand_forecast") < 0, 1
        ).otherwise(0)
    ).alias("negative_forecasts"),
    F.sum(
        F.when(
            F.isnan("demand_forecast"), 1
        ).otherwise(0)
    ).alias("nan_forecasts"),
    F.sum(
        F.when(F.col("inventory_risk_score").isNull(), 1).otherwise(0)
    ).alias("null_risk_scores"),
    F.sum(
        F.when(
            (F.col("inventory_risk_score") < 0)
            | (F.col("inventory_risk_score") > 100),
            1
        ).otherwise(0)
    ).alias("out_of_range_scores"),
    F.sum(
        F.when(
            ~F.col("inventory_risk_category").isin("LOW", "MEDIUM", "HIGH")
            | F.col("inventory_risk_category").isNull(),
            1
        ).otherwise(0)
    ).alias("invalid_categories"),
    F.sum(
        F.when(
            F.col("safety_stock_proxy") < 0, 1
        ).otherwise(0)
    ).alias("negative_safety_stock"),
    F.sum(
        F.when(
            F.col("replenishment_target") < 0, 1
        ).otherwise(0)
    ).alias("negative_replenishment")
).first().asDict()

print("\n--- FORECAST VALIDITY ---")
for check, value in numeric_checks.items():
    passed = value == 0
    print(f"[{'PASS' if passed else 'FAIL'}] {check}: {value:,}")

# 5. Check duplicate forecast keys
duplicate_keys = (
    forecast
    .groupBy("forecast_date", "item_id", "store_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"\n[{'PASS' if duplicate_keys == 0 else 'FAIL'}] Duplicate forecast keys: {duplicate_keys:,}")

# 6. Check missing historical seasonal lags, if saved in forecast
lag_columns = ["sales_7", "sales_14", "sales_21", "sales_28"]

print("\n--- MISSING SEASONAL LAGS ---")
lag_null_counts = {}

for column in lag_columns:
    if column in forecast.columns:
        count = forecast.filter(F.col(column).isNull()).count()
        lag_null_counts[column] = count
        print(f"{column}: {count:,} missing")
    else:
        print(f"{column}: not retained in saved forecast; cannot verify")

# 7. Risk category summary
print("\n--- INVENTORY RISK SUMMARY ---")
forecast.groupBy("inventory_risk_category").agg(
    F.count("*").alias("rows"),
    F.round(F.sum("demand_forecast"), 2).alias("forecasted_units"),
    F.round(F.avg("inventory_risk_score"), 2).alias("avg_risk_score"),
    F.round(F.avg("safety_stock_proxy"), 2).alias("avg_safety_stock")
).orderBy("inventory_risk_category").show(truncate=False)

# 8. Historical sales baseline
print("--- HISTORICAL SALES BASELINE ---")
spark.table("gold_fact_sales").agg(
    F.count("*").alias("historical_rows"),
    F.sum("sales").alias("historical_units"),
    F.min("date").alias("history_start"),
    F.max("date").alias("history_end")
).show(truncate=False)

# 9. Backtest metrics: use variables only if still in memory.
# These are not recomputed from future predictions because actual future
# observations are unavailable in gold_demand_forecast.
print("--- BACKTEST METRICS ---")
try:
    metrics_available = all(
        math.isfinite(float(value)) and float(value) >= 0
        for value in (mae, rmse, wape)
    )
    if metrics_available:
        print(f"MAE:  {float(mae):.4f}")
        print(f"RMSE: {float(rmse):.4f}")
        print(f"WAPE: {float(wape):.2f}%")
        print("[PASS] Backtest metrics available in notebook session")
    else:
        print("[FAIL] Backtest metrics are not finite/non-negative")
except NameError:
    print("[NOT VERIFIED] MAE/RMSE/WAPE variables are unavailable.")
    print("Recompute them from saved backtest predictions and actuals.")

# 10. Overall verdict
critical_failures = []

if missing_tables:
    critical_failures.append("Required tables missing")
if row_count == 0:
    critical_failures.append("Forecast table is empty")
if not horizon_pass:
    critical_failures.append("Forecast horizon is not 28 consecutive dates")
if not date_alignment_pass:
    critical_failures.append("Forecast dates do not align with historical cutoff")
if duplicate_keys != 0:
    critical_failures.append("Duplicate forecast keys found")

for check, value in numeric_checks.items():
    if value != 0:
        critical_failures.append(f"{check} is non-zero")

print("\n" + "=" * 65)
if critical_failures:
    print("VERDICT: FAIL — INVESTIGATE BEFORE CLAIMING DAY 5 COMPLETE")
    for failure in critical_failures:
        print(" -", failure)
else:
    print("VERDICT: STRUCTURAL CHECKS PASSED")
    print("Review missing seasonal lags and backtest metrics above.")
    print("This validates the saved output, not forecast accuracy by itself.")
print("=" * 65)


StatementMeta(, e8bd53c3-c827-4a09-ac99-0f2217e5078b, 48, Finished, Available, Finished, False)

DAY 5 — DEMAND FORECASTING & INVENTORY INTELLIGENCE AUDIT
[PASS] Table: gold_dim_date
[PASS] Table: gold_dim_product
[PASS] Table: gold_dim_store
[PASS] Table: gold_fact_sales
[PASS] Table: gold_data_quality_audit
[PASS] Table: gold_demand_forecast

--- FORECAST COVERAGE ---
Rows:             853,720
Forecast dates:   28
Item-store series:    30,490
Start date:       2016-04-25
End date:         2016-05-22
[PASS] 28 consecutive forecast dates
Historical last date: 2016-04-24
[PASS] Forecast starts the day after historical data

--- FORECAST VALIDITY ---
[PASS] null_forecasts: 0
[PASS] negative_forecasts: 0
[PASS] nan_forecasts: 0
[PASS] null_risk_scores: 0
[PASS] out_of_range_scores: 0
[PASS] invalid_categories: 0
[PASS] negative_safety_stock: 0
[PASS] negative_replenishment: 0

[PASS] Duplicate forecast keys: 0

--- MISSING SEASONAL LAGS ---
sales_7: 640,290 missing
sales_14: 426,860 missing
sales_21: 213,430 missing
sales_28: 0 missing

--- INVENTORY RISK SUMMARY ---
+---------------